# 🍍 Pineapple Cup Detection — Interactive Walkthrough

This notebook is a **step-by-step guide** that you can run yourself.  
Each section explains what is happening, then lets you run the code directly.

![Grid overview](docs/grid_overview.png)

---

## What does this system do?

Given a photo of a 3×3 bookshelf, it tells you:
- **Is the pineapple cup in the photo?**
- **Which shelf cell is it in?**

The 9 cells are numbered like this:

```
| 0 | 1 | 2 |   ← top row
| 3 | 4 | 5 |   ← middle row
| 6 | 7 | 8 |   ← bottom row
```

Output example:
```json
{ "has_cup": 1, "cell_id": 4 }
```

---

## How the whole pipeline works

| Step | What happens |
|---|---|
| 1. Look at the images | Understand what the cup looks like and where it appears |
| 2. Verify annotations | Check the bounding boxes drawn in Label Studio are correct |
| 3. Augment | Generate ~260 training images from 10 originals |
| 4. Train | Teach YOLOv8 to detect the cup |
| 5. Detect | Run the trained model on a new photo |
| 6. Evaluate | Check accuracy across all 10 sample images |

---
## Step 0 — Install dependencies

Run this once when setting up for the first time.

In [ ]:
import subprocess, sys
subprocess.check_call(["uv", "sync"], cwd=".")
print("✅ Dependencies installed")

---
## Step 1 — Look at the sample images

We have 10 reference photos:
- **9 images** — cup present, one per shelf cell
- **1 image** — `missing.png`, no cup at all

Let's display all of them so you can see what we're working with.

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

img = mpimg.imread("docs/grid_overview.png")
fig, ax = plt.subplots(figsize=(12, 13))
ax.imshow(img)
ax.axis("off")
ax.set_title(
    "All 9 cup positions — each photo shows the cup in a different shelf cell",
    fontsize=13, pad=12
)
plt.tight_layout()
plt.show()

---
## Step 2 — Verify annotations

After you export annotations from Label Studio, run this to visually confirm every bounding box is correct.

The grid overlay shows the 9 cell boundaries.  
The **green box** should tightly surround the pineapple cup.  
`missing.png` should have no green box.

In [ ]:
import subprocess
result = subprocess.run([
    "python", "annotate.py",
    "--src_images", "sample_data/",
    "--src_labels", "annotations/",
    "--out_dir", "annotation_preview/",
    "--no_show"
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("⚠️  Error:", result.stderr)

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from pathlib import Path

previews = sorted(Path("annotation_preview").glob("*.png"))

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for ax, p in zip(axes, previews):
    ax.imshow(mpimg.imread(p))
    ax.set_title(p.stem.replace("_preview", ""), fontsize=10)
    ax.axis("off")

plt.suptitle("Annotation previews — green box = cup, red lines = grid", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Step 3 — Augment the dataset

We only have 10 photos. That is not enough to train a model.  
This step generates **~260 variations** of those 10 images by randomly adjusting brightness, blur, colour, and small shifts.

> The model never sees the exact same photo twice — this helps it generalise to new photos.

In [ ]:
import subprocess
result = subprocess.run([
    "python", "augment.py",
    "--src_images", "sample_data/",
    "--src_labels", "annotations/",
    "--out_dir", "dataset/",
    "--n_aug", "25"
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("⚠️  Error:", result.stderr)

In [ ]:
# Show a sample of augmented variants from one original image
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from pathlib import Path

aug_images = sorted(Path("dataset/images/train").glob("top_left*.png"))[:6]

fig, axes = plt.subplots(1, 6, figsize=(20, 4))
for ax, p in zip(axes, aug_images):
    ax.imshow(mpimg.imread(p))
    ax.set_title(p.stem[-8:], fontsize=8)
    ax.axis("off")

plt.suptitle("6 augmented variants of top_left.png", fontsize=12)
plt.tight_layout()
plt.show()

---
## Step 4 — Train the model

This teaches **YOLOv8-nano** (a small, fast object detection model) to recognise the pineapple cup.

| Setting | Value | Why |
|---|---|---|
| Model | YOLOv8-nano | Smallest variant — good for a small dataset |
| Epochs | 100 | Max training rounds (stops early if no improvement) |
| Batch size | 8 | How many images processed at once |
| Early stopping | 20 epochs | Stops if accuracy doesn't improve — saves time |

⏱️ **Expected time: 10–30 minutes on CPU.**

In [ ]:
import subprocess
result = subprocess.run(
    ["python", "train.py"],
    capture_output=False,  # stream output live to the notebook
    text=True
)
if result.returncode != 0:
    print("⚠️  Training failed")

In [ ]:
# Show the training loss/metric curves
from pathlib import Path
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

results_img = Path("runs/train/pineapple_cup_v1/results.png")
if results_img.exists():
    plt.figure(figsize=(16, 6))
    plt.imshow(mpimg.imread(results_img))
    plt.axis("off")
    plt.title("Training curves", fontsize=13)
    plt.show()
else:
    print("Training results image not found — has training completed?")

---
## Step 5 — Detect the cup in a single image

Now we use the trained model to detect the cup in one photo and see the result.

Change `test_image` below to try different images.

In [ ]:
import json, sys
sys.path.insert(0, ".")
from detect import Detector

WEIGHTS = "runs/train/pineapple_cup_v1/weights/best.pt"
test_image = "sample_data/top_left.png"   # ← change this to any image

detector = Detector(WEIGHTS)
result = detector.detect(test_image)

print(f"Image : {test_image}")
print(f"Result: {json.dumps(result, indent=2)}")

CELL_NAMES = [
    "top-left",    "top-center",    "top-right",
    "middle-left", "middle-center", "middle-right",
    "bottom-left", "bottom-center", "bottom-right",
]
if result["has_cup"]:
    print(f"\n✅ Cup found in cell {result['cell_id']} ({CELL_NAMES[result['cell_id']]})")
else:
    print("\n❌ No cup detected")

In [ ]:
# Show the annotated result image with bounding box and grid overlay
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import cv2

result, annotated_bgr = detector.detect_with_viz(test_image)
annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(7, 7))
plt.imshow(annotated_rgb)
plt.axis("off")
title = f"has_cup={result['has_cup']}  cell_id={result['cell_id']}"
plt.title(title, fontsize=13)
plt.show()

---
## Step 6 — Evaluate on all 10 sample images

This is the final accuracy check.  
We run the model on every sample image and compare predictions to the known ground truth.

**What we measure:**

| Metric | Meaning |
|---|---|
| Presence accuracy | How often does it correctly say cup / no cup? |
| Cell accuracy | How often does it pick the right shelf cell? |
| End-to-end accuracy | Both presence AND cell correct |

In [ ]:
from evaluate import evaluate
metrics = evaluate(WEIGHTS, data_dir="sample_data", save_viz=True)

In [ ]:
# Display all 10 result images side by side
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

viz_images = sorted(Path("eval_viz").glob("*.png"))

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

for ax, p in zip(axes, viz_images):
    img = cv2.imread(str(p))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(p.stem.replace("_eval", ""), fontsize=9)
    ax.axis("off")

plt.suptitle("Evaluation results — green box = detection, highlighted cell = predicted", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Summary bar chart
import matplotlib.pyplot as plt

labels = ["Presence", "Cell", "End-to-end"]
values = [
    metrics["presence_accuracy"] * 100,
    metrics["cell_accuracy"] * 100,
    metrics["end_to_end_accuracy"] * 100,
]
colors = ["#4CAF50" if v == 100 else "#FF9800" for v in values]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, values, color=colors, edgecolor="white", width=0.5)
ax.set_ylim(0, 110)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Model accuracy on 10 sample images")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{val:.0f}%", ha="center", fontsize=13, fontweight="bold")
ax.axhline(100, color="gray", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

---
## Done! 🎉

You have successfully:

| ✅ | What you did |
|---|---|
| ✅ | Labelled 10 images with bounding boxes |
| ✅ | Generated ~260 training images via augmentation |
| ✅ | Trained a YOLOv8 model to detect the pineapple cup |
| ✅ | Tested it on a single image |
| ✅ | Evaluated accuracy across all 10 sample images |

---

### Next steps

- **More data** — photograph the cup in different lighting conditions for a more robust model
- **Upgrade model** — swap `yolov8n.pt` → `yolov8s.pt` in `train.py` for better accuracy
- **Deploy** — import `Detector` from `detect.py` into any Python application